In [ ]:
!pip install "transformers" "transformers[torch]" -q 

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [ ]:
train_data = pd.read_csv("/kaggle/input/datasets/nileshmalode1/samsum-dataset-text-summarization/samsum-train.csv")
val_data = pd.read_csv("/kaggle/input/datasets/nileshmalode1/samsum-dataset-text-summarization/samsum-validation.csv")

In [ ]:
train_data.head()

In [ ]:
train_data.shape

In [ ]:
val_data.shape

In [ ]:
# random sampling => do if less computation

# train_data = train_data.sample(n=4000, random_state=42).reset_index(drop = True)
# val_data = val_data.sample(n=500, random_state=42).reset_index(drop = True)

# Drop any rows where dialogue or summary is missing
train_data = train_data.dropna().reset_index(drop=True)
val_data = val_data.dropna().reset_index(drop=True)

# Data pre-processing

In [ ]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text)  # lines
    text = re.sub(r"\s+", " ", text)   # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags <p> <h1>
    text = text.strip().lower()
    
    return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [ ]:
train_data["dialogue"][0]

# Tokenize

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

# t5 = 32k vocabulary

In [ ]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets ["input_ids"] # token ids => add to input as labels

    return inputs

In [ ]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [ ]:
train_dataset[0]

### input ids - dialogue => token ids
### 1 => EOS; 0 => padding 

### attention mask

### labels - target => summary token

In [ ]:
len(train_dataset[0]["input_ids"])

In [ ]:
type(train_dataset)
type(val_dataset)

# Working with our Model

In [ ]:
# NLP => generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

## fine-tune

In [ ]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ", device)
model.to(device)

In [ ]:
# Training Arguements

training_args = TrainingArguments(
    
    output_dir = "./results",
    
    num_train_epochs = 6,
    weight_decay = 0.01,
    
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500
    #0 => lr default
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [ ]:
# train the model
trainer.train()

In [ ]:
# save the trained model

model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

In [ ]:
# load model

model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

# Test the core logic for summarization

In [ ]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [ ]:
test_dialogue_4 = """
Alex: Hey everyone, are we still doing the cabin trip this weekend?
Jamie: Yes! I am so ready to get out of the city. I need a break.
Taylor: I'm still in, but there's a slight problem. I can't leave until Friday night. I got scheduled for a late meeting.
Alex: Oh, that's fine. Jamie and I can head up early on Friday afternoon to get the keys from the host and buy all the groceries.
Morgan: Wait, I thought I was riding with Jamie? 
Jamie: You are! I have an SUV so there's enough room for you, Alex, and all the bags. We can swing by your place and pick you up around 2 PM.
Morgan: Awesome. What about the food situation? Should we all just split the cost?
Alex: Yeah, since Jamie and I are doing the shopping, I'll just keep the receipt. Everyone just Venmo me $40 by Thursday so we have a budget.
Taylor: Sounds good. I'll just drive up by myself on Friday night around 8 PM. Can someone send me the exact address?
Alex: Just dropped the pin in the chat. Oh hey Taylor, since you're coming later, can you bring some extra firewood? The host said they are running low and the nights get freezing.
Taylor: No problem at all. I'll grab a few bundles from the gas station on the way up.
Jamie: Perfect. So the official plan: Me, Alex, and Morgan leave at 2 PM on Friday. We buy groceries. Taylor comes at 8 PM with firewood. 
Morgan: Don't forget your sleeping bags! See you all Friday.
"""

print("Model Summary:", summarize_dialogue(test_dialogue_4))